In [2]:
# ============================================================
# 02_ml_underwriting.ipynb
# XGBoost Underwriting Model — Hypertension / Diabetes /
# Dyslipidemia
#
# Paper: Adverse Selection Risk and Counterfactual Explanation
#        Guardrails in Healthcare AI Underwriting:
#        A Quantitative Governance Evaluation under
#        Korea's AI Basic Act
#
# Pipeline:
#   1. Load preprocessed CSVs
#   2. Prepare actionable feature matrix (non-actionable excluded)
#   3. Stratified 80/20 train-test split
#   4. Optuna HPO with 5-fold inner CV (per disease × subgroup)
#   5. Refit final model on full training set
#   6. Evaluate on held-out test set
#   7. SHAP global importance
#   8. Save models + results
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Library Imports
# ─────────────────────────────────────────────
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss
)

import xgboost as xgb
import optuna
import shap
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Libraries loaded successfully.")
print(f"  XGBoost : {xgb.__version__}")
print(f"  Optuna  : {optuna.__version__}")
print(f"  SHAP    : {shap.__version__}")


# ─────────────────────────────────────────────
# Cell 2 | Path Configuration
# ─────────────────────────────────────────────
BASE_OUT = os.path.join('..', 'outputs')
BASE_MOD = os.path.join('..', 'outputs', 'models')
BASE_FIG = os.path.join('..', 'outputs', 'figures')

for d in [BASE_MOD, BASE_FIG]:
    os.makedirs(d, exist_ok=True)

print("Output directories ready.")


# ─────────────────────────────────────────────
# Cell 3 | Dataset Registry
# ─────────────────────────────────────────────
DISEASE_REGISTRY = {
    'htn': {
        'target': 'HE_HP',
        'label' : 'Hypertension',
    },
    'dm': {
        'target': 'HE_DM_HbA1c',
        'label' : 'Diabetes',
    },
    'dys': {
        'target': 'HE_HCHOL',
        'label' : 'Dyslipidemia',
    },
}

SUBGROUPS = ['total', 'male', 'female']

def load_dataset(disease_key, subgroup):
    path = os.path.join(BASE_OUT, f"{disease_key}_{subgroup}.csv")
    return pd.read_csv(path)

print("Dataset registry configured.")


# ─────────────────────────────────────────────
# Cell 4 | Feature Engineering Helper
# ─────────────────────────────────────────────
# Non-actionable columns excluded from the feature matrix.
# These correspond to CFE constraints phi_imm and phi_caus
# established in the preprocessing stage.
# age & sex are retained in the CSV for RAG stratification
# but excluded here as model inputs.

NON_ACTIONABLE = [
    # Identifiers
    'ID', 'year',
    # Derived stratification variable
    'age_group',
    # phi_imm: demographic
    'age', 'sex',
    # phi_imm: socioeconomic (fixed short-term)
    'edu', 'incm', 'ho_incm',
    # phi_imm: occupation-dependent PA
    'BE3_71', 'BE3_81',
    # phi_imm: psychological state
    'BP1', 'mh_stress',
    # phi_caus: derived from BMI/weight
    'HE_obe',
    # phi_imm: past weight-change records
    'BO1_1', 'BO1_2', 'BO1_3',
]

def prepare_Xy(df, target_col):
    drop_cols    = NON_ACTIONABLE + [target_col]
    feature_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feature_cols].copy()
    y = df[target_col].astype(int).copy()
    return X, y, feature_cols


# ─────────────────────────────────────────────
# Cell 5 | English Feature Label Map
# ─────────────────────────────────────────────
FEATURE_LABEL_MAP = {
    # Anthropometrics
    'HE_BMI'    : 'BMI (kg/m²)',
    'HE_wc'     : 'Waist circumference (cm)',
    'HE_wt'     : 'Body weight (kg)',
    # Substance use
    'BS3_1'     : 'Current smoking',
    'BD1_11'    : 'Alcohol drinking frequency',
    'BD2_1'     : 'Alcohol amount per occasion',
    # Physical activity (leisure & transport only)
    'BE3_75'    : 'High-intensity PA: leisure',
    'BE3_91'    : 'PA: transportation',
    'pa_aerobic': 'Aerobic PA practice',
    # Diet behaviour
    'L_BR_FQ'   : 'Breakfast frequency',
    # Healthcare utilisation
    'BH1'       : 'Health checkup (past year)',
    # HTN-specific diet
    'N_NA'      : 'Sodium intake (mg)',
    'N_K'       : 'Potassium intake (mg)',
    # DM-specific diet
    'N_SUGAR'   : 'Sugar intake (g)',
    'N_CHO'     : 'Carbohydrate intake (g)',
    'N_EN'      : 'Energy intake (kcal)',
    # DYS-specific diet
    'N_FAT'     : 'Fat intake (g)',
    'N_CHOL'    : 'Dietary cholesterol (mg)',
}

def to_labels(feature_cols):
    return [FEATURE_LABEL_MAP.get(c, c) for c in feature_cols]


# ─────────────────────────────────────────────
# Cell 6 | Optuna Objective (inner CV)
# ─────────────────────────────────────────────
N_TRIALS    = 50
N_CV_INNER  = 5
RANDOM_SEED = 42

def make_objective(X_tr, y_tr, scale_pos_weight):
    def objective(trial):
        params = {
            'verbosity'        : 0,
            'objective'        : 'binary:logistic',
            'eval_metric'      : 'auc',
            'tree_method'      : 'hist',
            'random_state'     : RANDOM_SEED,
            'scale_pos_weight' : scale_pos_weight,
            'n_estimators'     : trial.suggest_int(
                                     'n_estimators', 200, 1000, step=100),
            'max_depth'        : trial.suggest_int('max_depth', 3, 8),
            'learning_rate'    : trial.suggest_float(
                                     'learning_rate', 0.01, 0.2, log=True),
            'subsample'        : trial.suggest_float(
                                     'subsample', 0.6, 1.0),
            'colsample_bytree' : trial.suggest_float(
                                     'colsample_bytree', 0.6, 1.0),
            'min_child_weight' : trial.suggest_int(
                                     'min_child_weight', 1, 10),
            'reg_alpha'        : trial.suggest_float(
                                     'reg_alpha', 1e-4, 10.0, log=True),
            'reg_lambda'       : trial.suggest_float(
                                     'reg_lambda', 1e-4, 10.0, log=True),
            'gamma'            : trial.suggest_float('gamma', 0.0, 5.0),
        }

        skf  = StratifiedKFold(n_splits=N_CV_INNER,
                               shuffle=True,
                               random_state=RANDOM_SEED)
        aucs = []
        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            X_t, X_v = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
            y_t, y_v = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]
            model = xgb.XGBClassifier(**params)
            model.fit(X_t, y_t,
                      eval_set=[(X_v, y_v)],
                      verbose=False)
            prob = model.predict_proba(X_v)[:, 1]
            aucs.append(roc_auc_score(y_v, prob))
        return np.mean(aucs)

    return objective


# ─────────────────────────────────────────────
# Cell 7 | Evaluation Helper
# ─────────────────────────────────────────────
def evaluate_model(model, X_test, y_test, run_label=''):
    prob = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)

    auroc = roc_auc_score(y_test, prob)
    auprc = average_precision_score(y_test, prob)
    brier = brier_score_loss(y_test, prob)
    cm    = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    sens  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec  = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"\n  {'─'*52}")
    print(f"  Evaluation: {run_label}")
    print(f"  {'─'*52}")
    print(f"  AUROC       : {auroc:.4f}")
    print(f"  AUPRC       : {auprc:.4f}")
    print(f"  Brier Score : {brier:.4f}")
    print(f"  Sensitivity : {sens:.4f}   "
          f"Specificity : {spec:.4f}")
    print(f"  Confusion Matrix "
          f"(rows=Actual, cols=Predicted):")
    print(f"               Pred 0   Pred 1")
    print(f"  Actual 0 :  {tn:6d}   {fp:6d}")
    print(f"  Actual 1 :  {fn:6d}   {tp:6d}")

    return {
        'AUROC'      : round(auroc, 4),
        'AUPRC'      : round(auprc, 4),
        'Brier'      : round(brier, 4),
        'Sensitivity': round(sens,  4),
        'Specificity': round(spec,  4),
        'TP': int(tp), 'FP': int(fp),
        'FN': int(fn), 'TN': int(tn),
    }


# ─────────────────────────────────────────────
# Cell 8 | SHAP Global Importance Helper
# ─────────────────────────────────────────────
def save_shap_summary(model, X_test, feature_cols,
                      disease_key, subgroup):
    eng_labels  = to_labels(feature_cols)
    X_disp      = X_test.copy()
    X_disp.columns = eng_labels

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_disp)
    disease_lbl = DISEASE_REGISTRY[disease_key]['label']

    # Beeswarm
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_disp,
                      feature_names=eng_labels,
                      show=False, max_display=20)
    plt.title(
        f"SHAP Feature Importance — "
        f"{disease_lbl} ({subgroup})",
        fontsize=13, pad=12
    )
    plt.xlabel("SHAP value (impact on log-odds of positive class)",
               fontsize=11)
    plt.tight_layout()
    bee_path = os.path.join(
        BASE_FIG,
        f"shap_beeswarm_{disease_key}_{subgroup}.png"
    )
    plt.savefig(bee_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Mean |SHAP| bar
    mean_abs = pd.DataFrame({
        'feature_code' : feature_cols,
        'feature_label': eng_labels,
        'mean_abs_shap': np.abs(shap_values).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False
     ).reset_index(drop=True)

    plt.figure(figsize=(9, 6))
    plt.barh(
        mean_abs['feature_label'].head(20)[::-1],
        mean_abs['mean_abs_shap'].head(20)[::-1],
        color='steelblue'
    )
    plt.xlabel('Mean |SHAP value|', fontsize=11)
    plt.title(
        f"Mean Absolute SHAP — "
        f"{disease_lbl} ({subgroup})",
        fontsize=13
    )
    plt.tight_layout()
    bar_path = os.path.join(
        BASE_FIG,
        f"shap_bar_{disease_key}_{subgroup}.png"
    )
    plt.savefig(bar_path, dpi=150, bbox_inches='tight')
    plt.close()

    # CSV
    csv_path = os.path.join(
        BASE_FIG,
        f"shap_importance_{disease_key}_{subgroup}.csv"
    )
    mean_abs.to_csv(csv_path, index=False)

    print(f"    SHAP beeswarm → {bee_path}")
    print(f"    SHAP bar      → {bar_path}")
    print(f"    SHAP CSV      → {csv_path}")
    return mean_abs


# ─────────────────────────────────────────────
# Cell 9 | Main Training Loop
# ─────────────────────────────────────────────
all_results = {}

for d_key, d_info in DISEASE_REGISTRY.items():
    target = d_info['target']
    label  = d_info['label']
    all_results[d_key] = {}

    print(f"\n{'='*58}")
    print(f"  DISEASE : {label}  |  Target : {target}")
    print(f"{'='*58}")

    for sg in SUBGROUPS:
        print(f"\n  -- Subgroup : {sg} "
              + "-" * (42 - len(sg)))

        # Load & prepare
        df = load_dataset(d_key, sg)
        X, y, feature_cols = prepare_Xy(df, target)

        # Stratified 80/20 split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.2,
            stratify=y,
            random_state=RANDOM_SEED
        )
        print(f"    Train    : {len(X_train):,}  "
              f"Test : {len(X_test):,}  "
              f"Features : {len(feature_cols)}")
        print(f"    Features : {feature_cols}")

        # Class weight
        n0  = (y_train == 0).sum()
        n1  = (y_train == 1).sum()
        spw = round(n0 / n1, 4)
        print(f"    Positive rate (train) : "
              f"{n1/(n0+n1)*100:.1f}%  "
              f"scale_pos_weight : {spw}")

        # Optuna HPO
        print(f"    Running Optuna "
              f"({N_TRIALS} trials, "
              f"{N_CV_INNER}-fold inner CV) ...")
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(
                seed=RANDOM_SEED)
        )
        study.optimize(
            make_objective(X_train, y_train, spw),
            n_trials=N_TRIALS,
            show_progress_bar=False
        )

        best_params = study.best_params
        best_params.update({
            'verbosity'        : 0,
            'objective'        : 'binary:logistic',
            'eval_metric'      : 'auc',
            'tree_method'      : 'hist',
            'random_state'     : RANDOM_SEED,
            'scale_pos_weight' : spw,
        })
        print(f"    Best inner CV AUROC : "
              f"{study.best_value:.4f}")
        print(f"    Best hyperparameters:")
        skip = {'verbosity', 'objective',
                'eval_metric', 'tree_method', 'random_state'}
        for k, v in best_params.items():
            if k not in skip:
                print(f"      {k:<22} : {v}")

        # Refit on full training set
        final_model = xgb.XGBClassifier(**best_params)
        final_model.fit(X_train, y_train, verbose=False)

        # Evaluate
        metrics = evaluate_model(
            final_model, X_test, y_test,
            run_label=f"{label} / {sg}"
        )
        metrics.update({
            'best_cv_auroc': round(study.best_value, 4),
            'best_params'  : best_params,
            'n_train'      : int(len(X_train)),
            'n_test'       : int(len(X_test)),
            'n_features'   : int(len(feature_cols)),
            'feature_cols' : feature_cols,
        })
        all_results[d_key][sg] = metrics

        # SHAP
        print(f"    Computing SHAP importance ...")
        shap_imp = save_shap_summary(
            final_model, X_test,
            feature_cols, d_key, sg
        )
        print(f"    Top-5 features by mean |SHAP|:")
        for _, row in shap_imp.head(5).iterrows():
            print(f"      {row['feature_label']:<35} "
                  f"{row['mean_abs_shap']:.4f}")

        # Save model
        model_path = os.path.join(
            BASE_MOD, f"xgb_{d_key}_{sg}.joblib"
        )
        joblib.dump(final_model, model_path)
        print(f"    Model saved : {model_path}")


# ─────────────────────────────────────────────
# Cell 10 | Results Summary Table
# ─────────────────────────────────────────────
rows = []
for d_key, d_info in DISEASE_REGISTRY.items():
    for sg in SUBGROUPS:
        m = all_results[d_key][sg]
        rows.append({
            'Disease'    : d_info['label'],
            'Subgroup'   : sg,
            'N_train'    : m['n_train'],
            'N_test'     : m['n_test'],
            'N_features' : m['n_features'],
            'CV_AUROC'   : m['best_cv_auroc'],
            'AUROC'      : m['AUROC'],
            'AUPRC'      : m['AUPRC'],
            'Brier'      : m['Brier'],
            'Sensitivity': m['Sensitivity'],
            'Specificity': m['Specificity'],
        })

df_results = pd.DataFrame(rows)

print("\n" + "="*85)
print("  MODEL PERFORMANCE SUMMARY")
print("="*85)
print(df_results.to_string(index=False))

res_path = os.path.join(BASE_OUT, 'model_results.csv')
df_results.to_csv(res_path, index=False)
print(f"\nPerformance table saved : {res_path}")


# ─────────────────────────────────────────────
# Cell 11 | Save Full Results (JSON)
# ─────────────────────────────────────────────
def serialise(obj):
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, list):
        return [serialise(i) for i in obj]
    if isinstance(obj, dict):
        return {k: serialise(v) for k, v in obj.items()}
    return obj

json_out = {}
for d_key, sg_dict in all_results.items():
    json_out[d_key] = {}
    for sg, metrics in sg_dict.items():
        m_clean = {k: v for k, v in metrics.items()
                   if k != 'feature_cols'}
        json_out[d_key][sg] = serialise(m_clean)

json_path = os.path.join(BASE_OUT, 'model_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_out, f, indent=2, ensure_ascii=False)
print(f"Full results (JSON) saved : {json_path}")

Libraries loaded successfully.
  XGBoost : 1.7.5
  Optuna  : 4.5.0
  SHAP    : 0.46.0
Output directories ready.
Dataset registry configured.

  DISEASE : Hypertension  |  Target : HE_HP

  -- Subgroup : total -------------------------------------
    Train    : 11,488  Test : 2,873  Features : 13
    Features : ['BD1_11', 'BD2_1', 'BS3_1', 'BE3_75', 'BE3_91', 'pa_aerobic', 'L_BR_FQ', 'BH1', 'HE_BMI', 'HE_wc', 'HE_wt', 'N_NA', 'N_K']
    Positive rate (train) : 32.2%  scale_pos_weight : 2.104
    Running Optuna (50 trials, 5-fold inner CV) ...
    Best inner CV AUROC : 0.7826
    Best hyperparameters:
      n_estimators           : 500
      max_depth              : 3
      learning_rate          : 0.02453143834695535
      subsample              : 0.6674869556303065
      colsample_bytree       : 0.7969055582218567
      min_child_weight       : 6
      reg_alpha              : 0.11457081589722987
      reg_lambda             : 0.012295667456423888
      gamma                  : 4.7683